### Case E-commerce Olist
Este desafio propõe a construção de um relatório executivo voltado a investidores e acionistas do setor de e-commerce, baseado no Brazilian E-Commerce Public Dataset by Olist. O objetivo é transformar dados transacionais em uma narrativa clara sobre desempenho comercial, eficiência logística e/ou satisfação do cliente, culminando em recomendações acionáveis e, quando possível, previsões fundamentadas.

In [5]:
# instalação das bibliotecas
pip install pandas matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-3.0.5-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached matplotlib-3.11.1-cp313-cp313-win_amd64.whl.metadata (80 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp313-cp313-win_amd64.whl.metadata (121 kB)
Using cached pandas-3.0.5-cp313-cp313-win_amd64.whl (9.8 MB)
Using cached matplotlib-3.11.1-cp313-cp313-win_amd64.whl (9.3 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.63.0-cp313-cp313-win_amd64.whl (2.3 MB)

   ---------------------------------------- 0/6 [fonttools]
   ---------------------------------------- 0/6 [fonttools]
   --------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\Michael\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
# importação das bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
# renomear arquivos, para facil acesso
customers = pd.read_csv("../data/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/olist_orders_dataset.csv")
products = pd.read_csv("../data/olist_products_dataset.csv")
sellers = pd.read_csv("../data/olist_sellers_dataset.csv")
categorys_translations = pd.read_csv("../data/product_category_name_translation.csv")

In [8]:
customers.head

<bound method NDFrame.head of                             customer_id                customer_unique_id  \
0      06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1      18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2      4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3      b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4      4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   
...                                 ...                               ...   
99436  17ddf5dd5d51696bb3d7c6291687be6f  1a29b476fee25c95fbafc67c5ac95cf8   
99437  e7b71a9017aa05c9a7fd292d714858e8  d52a67c98be1cf6a5c84435bd38d095d   
99438  5e28dfe12db7fb50a4b2f691faecea5e  e9f50caf99f032f0bf3c55141f019d99   
99439  56b18e2166679b8a959d72dd06da27f9  73c2643a0a458b49f58cea58833b192e   
99440  274fa6071e5e17fe303b9748641082c8  84732c5050c01db9b23e19ba39899398   

       customer_zip_code_prefix          cust

In [9]:
# converter datas de string para date
# lista de colunas de data na tabela de pedidos
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# convertendo para datetime
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

# confirmando a alteração
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [13]:
# verificando os status dos pedidos
print("Status dos Pedidos")
print(orders['order_status'].value_counts())

print("\nValores Nulos na Tabela de Pedidos")
print(orders.isnull().sum())

Status dos Pedidos
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Valores Nulos na Tabela de Pedidos
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


Conforme o código acima, temos:
    - mais de 90 mil pedidos entregues;
    - um pouco mais de mil pedidos enviados atualmente;
    - 625 pedidos cancelados;
    - um pouco mais de 600 pedidos indiponiveis;
    - podemos ter algumas datas de entrega com valores nulos.



In [16]:
# unindo todas as tabelas no em uma variavel 
df_principal = orders.merge(order_items, on='order_id', how='left') \
                     .merge(order_payments, on='order_id', how='left') \
                     .merge(customers, on='customer_id', how='left') \
                     .merge(order_reviews, on='order_id', how='left') \
                     .merge(products, on='product_id', how='left') \
                     .merge(sellers, on='seller_id', how='left')

# o '.merge' é o PROCV do pandas(python), permite que podemos combinar dataframes.

In [19]:
# filtrando só pedidos entregue para analisar logística e satisfação dos clientes
df_entregues = df_principal[df_principal['order_status'] == 'delivered'].copy()
df_entregues.head

<bound method NDFrame.head of                                 order_id                       customer_id  \
0       e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1       e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
2       e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
3       53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
4       47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
...                                  ...                               ...   
119138  63943bddc261676b46f01ca7ac2f7bd8  1fca14ff2861355f6e5f14306ff977a7   
119139  83c1379a015df1e13d02aae0204711ab  1aa71eb042121263aafbe80c1b562c9c   
119140  11c177c8e97725db2631073c19f07b62  b331b74b18dc79bcdf6532d51e1637c1   
119141  11c177c8e97725db2631073c19f07b62  b331b74b18dc79bcdf6532d51e1637c1   
119142  66dea50a8b16d9b4dee7af250b4be1a5  edb027a75a1449115f6b43211ae02a24   

       order_status order_purchas

In [26]:
# criando coluda de calculo do tempo de entrega (em dias)
df_entregues['tempo_entrega_dias'] = (
    df_entregues['order_delivered_customer_date'] - df_entregues['order_purchase_timestamp']
).dt.total_seconds() / (24 * 3600)
df_entregues['tempo_entrega_dias']

0          8.436574
1          8.436574
2          8.436574
3         13.782037
4          9.394213
            ...    
119138    22.193727
119139    24.859421
119140    17.086424
119141    17.086424
119142     7.674306
Name: tempo_entrega_dias, Length: 115723, dtype: float64

In [24]:
# criando coluna de calculo de atraso em relação à data estimada (em dias)
# se o valor for positivo, significa que atrasou
df_entregues['dias_atraso'] = (
    df_entregues['order_delivered_customer_date'] - df_entregues['order_estimated_delivery_date']
).dt.total_seconds() / (24 * 3600)
df_entregues['dias_atraso']

0         -7.107488
1         -7.107488
2         -7.107488
3         -5.355729
4        -17.245498
            ...    
119138    -1.265324
119139    -5.524803
119140   -20.018819
119141   -20.018819
119142   -17.452431
Name: dias_atraso, Length: 115723, dtype: float64

In [27]:
# criando coluna indicadora de atraso (True = atrasou | False = no prazo)
df_entregues['foi_atrasado'] = df_entregues['dias_atraso'] > 0
df_entregues['foi_atrasado']

0         False
1         False
2         False
3         False
4         False
          ...  
119138    False
119139    False
119140    False
119141    False
119142    False
Name: foi_atrasado, Length: 115723, dtype: bool

In [28]:
print(f"Total de registros de compras entregues: {df_entregues.shape[0]}")

Total de registros de compras entregues: 115723


### Parte de Logistica
* **df_principal**: Tabela unificada contendo o cruzamento completo dos dados de pedidos, itens, pagamentos, clientes, avaliações, produtos e vendedores.
* **df_entregues**: Recorte contendo apenas os pedidos com status `delivered`, garantindo a consistência das análises de prazo.
* **Métricas Criadas**:
  * `tempo_entrega_dias`: Tempo total com tempo corrido entre a realização do pedido e a entrega ao cliente final.
  * `dias_atraso`: Diferença entre a data real de entrega e a data estimada original.
  * `foi_atrasado`: Parte booleana identificando compras que ultrapassaram a data limite combinada.

In [30]:
# média da nota de avaliação, comparando pedidos que foram entregue no prazo e pedidos atrasados
resumo_satisfacao = df_entregues.groupby('foi_atrasado')['review_score'].agg(['count', 'mean', 'median']).reset_index()

# renomeando as colunas e mostrando
resumo_satisfacao['Status'] = resumo_satisfacao['foi_atrasado'].map({False: 'No Prazo', True: 'Com Atraso'})
resumo_satisfacao = resumo_satisfacao.rename(columns={
    'count': 'Qtd Pedidos', 
    'mean': 'Nota Média', 
    'median': 'Nota Mediana'
})

resumo_satisfacao[['Status', 'Qtd Pedidos', 'Nota Média', 'Nota Mediana']]

,Status,Qtd Pedidos,Nota Média,Nota Mediana
0,No Prazo,106000,4.208736,5.0
1,Com Atraso,8862,2.546491,2.0


### Entrega x Satisfação
A análise comparativa revela um impacto significativo do cumprimento de prazos sobre a percepção do cliente:
* **Entregas no Prazo**: Mantêm uma nota média elevada próxima de 4.2 estrelas.
* **Entregas com Atraso**: Sofrem uma queda drástica na nota média caindo para próximo de 2.0 estrelas.

> **Conclusão de Negócio para Investidores**: A ineficiência logística não é apenas um custo operacional, mas um fator direto de insatisfação do cliente e perda de reputação da marca.